In [ ]:
import os

%cd /home1/kelidari/

REPO_URL = f"https://github.com/Nikelroid/multimodal-sentiment-classification/"
REPO_NAME = "multimodal-sentiment-classification"

if not os.path.exists(REPO_NAME):
    print("\nCloning repository...")
    res = os.system(f"git clone {REPO_URL} {REPO_NAME} > /dev/null 2>&1")
    if res == 0:
        print("Successfully cloned!")
    %cd $REPO_NAME
else:
    print("\nRepository found. Pulling latest changes...")
    %cd $REPO_NAME
    os.system(f"git remote set-url origin {REPO_URL}")
    os.system("git pull")
    print("Done.")


## 1. Setup Environment & Trackers
We authenticate with Weights & Biases to ensure the pipeline logs progress precisely exactly as intended.

In [ ]:
import sys
import wandb

if "." not in sys.path:
    sys.path.append(".")

print("Authenticating W&B...")
wandb.login()


## 2. Ingest Real Datasets
Downloads the robust multimodal archives representing exact conditions in the real codebase.

In [ ]:
!python src/data/ingestion.py


## 3. Functionality Assessment (Architecture Pre-Test)
Before kicking off full multi-hour dataset loading, we test the core architectural fusion functionality to catch tensor mismatch bugs immediately.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoImageProcessor
from src.config import TEXT_MODEL_NAME, VISION_BACKBONE_NAME
from src.models.multimodal import MultimodalFusionNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing pre-test architecture evaluation on [{device}]...")

test_model = MultimodalFusionNet(
    text_model_name=TEXT_MODEL_NAME,
    vit_model_name=VISION_BACKBONE_NAME,
    use_audio=True
).to(device)

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
feature_extractor = AutoImageProcessor.from_pretrained(VISION_BACKBONE_NAME)

# Simulating a batch (Size=2) mimicking dataloaders precisely
test_txt = tokenizer(["Testing context.", "Secondary stream."], padding=True, truncation=True, return_tensors="pt").to(device)
test_img = torch.randn(2, 3, 224, 224).to(device)  # Random Image emulation
test_aud = torch.randn(2, 16000).to(device)       # Random Audio emulation

with torch.no_grad():
    test_logits = test_model(test_txt['input_ids'], test_txt['attention_mask'], test_img, test_aud)

print(f"Logits resolved successfully! Final projection geometry: {list(test_logits.shape)}")
assert test_logits.shape == (2, 3), "Functional Test Failed: Dimensions deviated from expected num_classes projection."
print("Functionality Test VERIFIED. The codebase core components operate flawlessly!")

del test_model, test_txt, test_img, test_aud
torch.cuda.empty_cache()


## 4. Execute Real World Training
Firing the primary model using W&B loggers natively inside the pipeline.

In [ ]:
from src.pipelines.train import train

print("Triggering Training Event...")
train()


## 5. End-to-End Test Evaluation Loop
After the model exports `models/best_multimodal.pt`, we independently configure identical dataloaders for the test construct and execute `evaluate_model` to verify final statistics.

In [ ]:
import torch
import os
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoImageProcessor

import src.config
from src.data.dataloaders import MultimodalDataset
from src.data.preprocess import sent_preprocess
from src.models.multimodal import MultimodalFusionNet
from src.pipelines.train import collate_fn
from src.pipelines.evaluate import evaluate_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Loading Test evaluation datasets...")
test_dataset = MultimodalDataset(
    dataset_dir=src.config.MSCTD_DIR,
    images_dir="dataset/test/test_ende",
    texts_file="dataset/test/english_test.txt",
    sentiments_file="dataset/test/sentiment_test.txt",
    preprocess_text_func=sent_preprocess,
    audio_dir="AudioSample"
)

tokenizer = AutoTokenizer.from_pretrained(src.config.TEXT_MODEL_NAME)
feature_extractor = AutoImageProcessor.from_pretrained(src.config.VISION_BACKBONE_NAME)
collate = lambda b: collate_fn(b, tokenizer, feature_extractor)

test_loader = DataLoader(test_dataset, batch_size=src.config.BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=2)

print("Initializing standalone evaluation architecture...")
model = MultimodalFusionNet(
    text_model_name=src.config.TEXT_MODEL_NAME,
    vit_model_name=src.config.VISION_BACKBONE_NAME,
    use_audio=True
).to(device)

model_path = "models/best_multimodal.pt"
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Evaluation Checkpoint Mounted Natively: {model_path}")
else:
    print("Warning! No weights detected! Proceeding with baseline initialized distributions.")

print("\n--- Start E2E Target Evaluation ---")
acc, p, r, f1, cm = evaluate_model(model, test_loader, device)

print(f"Extracted Evaluation Accuracy  : {acc:.4f}")
print(f"Extracted Evaluation Precision : {p:.4f}")
print(f"Extracted Evaluation Recall    : {r:.4f}")
print(f"Extracted Evaluation F1 Score  : {f1:.4f}")
print("Confusion Matrix:\n", cm)
print("\nOperations Complete: Tested natively as executed in codebase!")
